In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
pd.set_option('display.max_columns', None)

## Load the dataset

In [3]:
df = pd.read_csv("Dataset/churn_data.csv")

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1 - 12
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25 - 36
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1 - 12
3,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37 - 48
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1 - 12


In [4]:
df = df.drop(columns=['tenure'])

In [5]:
summary_rows = []

n_rows = len(df)

for column_name, column_data in df.items():

    missing_count = column_data.isna().sum()
    missing_ratio = (missing_count / n_rows) * 100

    example_values = column_data.dropna().unique()[:3]
    example_preview = ", ".join(map(str, example_values))[:60]

    summary_rows.append({
        "column_name": column_name,
        "dtype": column_data.dtype,
        "unique_values": column_data.nunique(),
        "missing_count": missing_count,
        "missing_percent": round(missing_ratio, 2),
        "example_values": example_preview
    })

dataset_summary = pd.DataFrame(summary_rows)

display(
    dataset_summary.style.set_properties(**{"text-align": "left"})
)

,column_name,dtype,unique_values,missing_count,missing_percent,example_values
0,gender,object,2,0,0.000000,"Female, Male"
1,SeniorCitizen,object,2,0,0.000000,"No, Yes"
2,Partner,object,2,0,0.000000,"Yes, No"
3,Dependents,object,2,0,0.000000,"No, Yes"
4,PhoneService,object,2,0,0.000000,"No, Yes"
5,MultipleLines,object,3,0,0.000000,"No phone service, No, Yes"
6,InternetService,object,3,0,0.000000,"DSL, Fiber optic, No"
7,OnlineSecurity,object,3,0,0.000000,"No, Yes, No internet service"
8,OnlineBackup,object,3,0,0.000000,"Yes, No, No internet service"
9,DeviceProtection,object,3,0,0.000000,"No, Yes, No internet service"


## Separate features and target

In [6]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [7]:
y = y.map({
    "No": 0,
    "Yes": 1
})

In [8]:
print(y.value_counts())
print(y.value_counts(normalize=True))

Churn
0    5163
1    1869
Name: count, dtype: int64
Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64


## Identify numerical and categorical features

In [9]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [10]:
print("Numerical columns:")
print(numerical_cols)

print("\nCategorical columns:")
print(categorical_cols)

Numerical columns:
['MonthlyCharges', 'TotalCharges']

Categorical columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']


## Train-test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [12]:
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (5625, 19)
Testing set: (1407, 19)

Training target distribution:
Churn
0    0.734222
1    0.265778
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    0.734186
1    0.265814
Name: proportion, dtype: float64


## Create preprocessing pipeline

In [13]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [15]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [16]:
print("Original training features:", X_train.shape[1])
print("Processed training features:", X_train_processed.shape[1])

print("Original test features:", X_test.shape[1])
print("Processed test features:", X_test_processed.shape[1])

Original training features: 19
Processed training features: 51
Original test features: 19
Processed test features: 51


## Convert processed data into DataFrames

In [17]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))
print(feature_names)

Number of features: 51
['num__MonthlyCharges' 'num__TotalCharges' 'cat__gender_Female'
 'cat__gender_Male' 'cat__SeniorCitizen_No' 'cat__SeniorCitizen_Yes'
 'cat__Partner_No' 'cat__Partner_Yes' 'cat__Dependents_No'
 'cat__Dependents_Yes' 'cat__PhoneService_No' 'cat__PhoneService_Yes'
 'cat__MultipleLines_No' 'cat__MultipleLines_No phone service'
 'cat__MultipleLines_Yes' 'cat__InternetService_DSL'
 'cat__InternetService_Fiber optic' 'cat__InternetService_No'
 'cat__OnlineSecurity_No' 'cat__OnlineSecurity_No internet service'
 'cat__OnlineSecurity_Yes' 'cat__OnlineBackup_No'
 'cat__OnlineBackup_No internet service' 'cat__OnlineBackup_Yes'
 'cat__DeviceProtection_No' 'cat__DeviceProtection_No internet service'
 'cat__DeviceProtection_Yes' 'cat__TechSupport_No'
 'cat__TechSupport_No internet service' 'cat__TechSupport_Yes'
 'cat__StreamingTV_No' 'cat__StreamingTV_No internet service'
 'cat__StreamingTV_Yes' 'cat__StreamingMovies_No'
 'cat__StreamingMovies_No internet service' 'cat__Stream

In [18]:
type(X_train_processed), type(X_test_processed)

(numpy.ndarray, numpy.ndarray)

In [19]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

## Sanity Checks

In [20]:
print("Missing values in training data:")
print(X_train_processed_df.isnull().sum().sum())

print("\nMissing values in test data:")
print(X_test_processed_df.isnull().sum().sum())

Missing values in training data:
0

Missing values in test data:
0


In [21]:
print("X_train:", X_train_processed_df.shape)
print("X_test :", X_test_processed_df.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5625, 51)
X_test : (1407, 51)
y_train: (5625,)
y_test : (1407,)


## Save the preprocessor

In [22]:
import joblib

joblib.dump(
    preprocessor,
    "models/preprocessor.pkl"
)

['models/preprocessor.pkl']

## Save the split and preprocessed datasets

In [24]:
joblib.dump(X_train_processed, "Dataset/X_train.pkl")
joblib.dump(X_test_processed, "Dataset/X_test.pkl")
joblib.dump(y_train, "Dataset/y_train.pkl")
joblib.dump(y_test, "Dataset/y_test.pkl")

['Dataset/y_test.pkl']